# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kaylahiltermann/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/kaylahiltermann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [3]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [9]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [ ]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [11]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '5a64fe'. Skipping!
Property 'summary' already exists in node '56a752'. Skipping!
Property 'summary' already exists in node '7a5a06'. Skipping!
Property 'summary' already exists in node 'd14808'. Skipping!
Property 'summary' already exists in node 'c7219b'. Skipping!
Property 'summary' already exists in node 'f0937b'. Skipping!
Property 'summary' already exists in node 'c6c5e9'. Skipping!
Property 'summary' already exists in node '145b52'. Skipping!
Property 'summary' already exists in node '33e287'. Skipping!
Property 'summary' already exists in node '078173'. Skipping!
Property 'summary' already exists in node '5cb101'. Skipping!
Property 'summary' already exists in node 'abed40'. Skipping!
Property 'summary' already exists in node '7f5d1e'. Skipping!
Property 'summary' already exists in node '43b5a2'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '5a64fe'. Skipping!
Property 'summary_embedding' already exists in node '56a752'. Skipping!
Property 'summary_embedding' already exists in node 'd14808'. Skipping!
Property 'summary_embedding' already exists in node '5cb101'. Skipping!
Property 'summary_embedding' already exists in node '7f5d1e'. Skipping!
Property 'summary_embedding' already exists in node '7a5a06'. Skipping!
Property 'summary_embedding' already exists in node '33e287'. Skipping!
Property 'summary_embedding' already exists in node 'c7219b'. Skipping!
Property 'summary_embedding' already exists in node '145b52'. Skipping!
Property 'summary_embedding' already exists in node 'abed40'. Skipping!
Property 'summary_embedding' already exists in node '43b5a2'. Skipping!
Property 'summary_embedding' already exists in node 'f0937b'. Skipping!
Property 'summary_embedding' already exists in node 'c6c5e9'. Skipping!
Property 'summary_embedding' already exists in node '078173'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 480)

We can save and load our knowledge graphs as follows.

In [12]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 480)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

---

#### ✅ Answer #1:

1. SingleHopSpecificQuerySynthesizer - Creates questions that can be answered with ONE piece of information from ONE document
2. MultiHopAbstractQuerySynthesizer - Creates conceptual and thematic questions that need MULTIPLE pieces of information from MULTIPLE documents to answer using general principles, trends or concepts.
3. MultiHopSpecificQuerySynthesizer - Creates factual questions that need MULTIPLE pieces of information from MULTIPLE documents to answer using specific facts, numbers or names.

Finally, we can use our `TestSetGenerator` to generate our testset!

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
testset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)
testset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '1774cf'. Skipping!
Property 'summary' already exists in node 'ed211e'. Skipping!
Property 'summary' already exists in node '9f9eaf'. Skipping!
Property 'summary' already exists in node '21977a'. Skipping!
Property 'summary' already exists in node '5c260d'. Skipping!
Property 'summary' already exists in node 'e94839'. Skipping!
Property 'summary' already exists in node 'f92b1c'. Skipping!
Property 'summary' already exists in node 'b65d4d'. Skipping!
Property 'summary' already exists in node 'f1f984'. Skipping!
Property 'summary' already exists in node '214f0d'. Skipping!
Property 'summary' already exists in node '6a20cd'. Skipping!
Property 'summary' already exists in node 'dd4346'. Skipping!
Property 'summary' already exists in node '9a03d6'. Skipping!
Property 'summary' already exists in node '8df2f8'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is Chapter 2?,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not include informat...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) specify regarding we...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) pertains to weeks of instructi...,single_hop_specifc_query_synthesizer
2,What is Volume 8 about in relation to clinical...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study (FWS) program subjec...,[Non-Term Characteristics A program that measu...,"No, the Federal Work-Study (FWS) program is an...",single_hop_specifc_query_synthesizer
4,How do the requirements for disbursement timin...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour or non-term credit-hour programs...,multi_hop_abstract_query_synthesizer
5,"How do the regulatory citations, such as 34 CF...","[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulatory citations 34 CFR 668.3(a) and 3...,multi_hop_abstract_query_synthesizer
6,How do nonstandard terms and sequential course...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Nonstandard terms are defined as terms that do...,multi_hop_abstract_query_synthesizer
7,How credit hour allocation for clinical experi...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_abstract_query_synthesizer
8,Considering the detailed requirements outlined...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulations in Volume 2 specify that for c...,multi_hop_specific_query_synthesizer
9,Wha volume 2 and volume 8 discribe about the d...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 explains the academic year requiremen...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [20]:
from langsmith import Client

client = Client()

# Create unique dataset name with timestamp
dataset_name = f"Loan Synthetic Data - s07"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data for s07"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [22]:
for data_row in testset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [23]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [24]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [25]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [26]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [27]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [28]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

Finally, we can set-up our RAG LCEL chain!

In [30]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [31]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'Based on the provided context, the kinds of loans available include:\n\n- Direct Subsidized Loans (available only to undergraduate students)\n- Direct Unsubsidized Loans\n- Direct PLUS Loans (also referred to as student Federal PLUS Loans or parent PLUS Loans)\n- Subsidized and Unsubsidized Federal Stafford Loans (made under the Federal Family Education Loan (FFEL) Program before July 1, 2010)\n- Federal PLUS Loans (also made under the FFEL Program before July 1, 2010)\n\nNote that no new FFEL Program loans have been made since July 1, 2010. Graduate or professional students are eligible only for Direct Unsubsidized Loans and Direct PLUS Loans, not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4o as our evaluation LLM for our base Evaluators.

In [ ]:
eval_llm = ChatOpenAI(model="gpt-4o")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [ ]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

---

✅ Response:

- `qa_evaluator`: Evaluates the question-answering accuracy and factually correctness by comparing the generated answer against the reference answers. It is a binary score (correct or incorrect). 
- `labeled_helpfulness_evaluator`: Evaluates the helpfulness or unhelpfulness of the answer by determining subjectively whether the answer is useful or not to the user, using the reference answer as context for evaluation.
- `empathy_evaluator`: Evaluates the empathy of the response and whether it is emotionally supportive or not, subjectively determining whether it makes the user feel heard or not

## LangSmith Evaluation

In [35]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'memorable-drain-73' at:
https://smith.langchain.com/o/4e8d6936-58a0-4ff9-bfae-6484988e5784/datasets/e39f7268-9973-4abb-a78e-c519c003076e/compare?selectedSessions=23e7eedb-670e-4336-af4b-92a39633a547




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,disbursement appendix A or B how does that wor...,"Based on the provided context, Appendix B at t...",None,The disbursement timing rules are explained in...,0,0,0,6.126277,23b41cf0-8073-411f-bc04-9737009339dc,57ddb47d-5820-4e98-abbb-2999ecf3b104
1,How does the inclusion of clinical work in sta...,The inclusion of clinical work in standard ter...,None,The inclusion of clinical work in standard ter...,1,1,0,6.948081,e28bd36e-054b-4441-b723-d3b9d4637bc8,75919e24-6a76-4221-83b8-8f38a2b9f534
2,Wha volume 2 and volume 8 discribe about the d...,I don't know.,None,Volume 2 explains the academic year requiremen...,0,0,0,0.964745,40bbabbe-19d2-4a0a-bb13-ed17bc811df1,4aa6a325-4818-43a0-825e-a133e1d7741c
3,Considering the detailed requirements outlined...,Based on the provided context from Volume 2 an...,None,The regulations in Volume 2 specify that for c...,1,0,0,4.915865,866c5c78-eb89-4a97-97be-987d6cbaf8e0,c8314421-dc7f-4637-b62a-976c796a52a5
4,How credit hour allocation for clinical experi...,"Based on the provided context, clinical work t...",None,The context explains that clinical work includ...,1,1,0,4.991095,adf76b99-3f65-4bd8-9aca-d361f618b6cf,1d3b46c6-5575-4a9d-be41-bee644171c1a
5,How do nonstandard terms and sequential course...,Based on the provided context:\n\nNonstandard ...,None,Nonstandard terms are defined as terms that do...,1,1,0,11.215225,8b260132-4e1f-4572-b001-c3aafcde7c98,684fc4ad-79ca-4169-9cf3-10e5c3a08edf
6,"How do the regulatory citations, such as 34 CF...",The regulatory citations 34 CFR 668.3(a) and 3...,None,The regulatory citations 34 CFR 668.3(a) and 3...,1,1,0,3.507261,4f048b64-5a62-4fea-80bd-b02ea951bb3c,b092c20e-a5b7-4d5b-bbb4-776d523646d0
7,How do the requirements for disbursement timin...,The requirements for disbursement timing in cl...,None,In clock-hour or non-term credit-hour programs...,1,1,0,7.837631,13443210-8a92-4356-9514-111fe686f5bc,08c35991-2726-4411-8ae6-b7a5824c0bba
8,Is the Federal Work-Study (FWS) program subjec...,"No, the Federal Work-Study (FWS) program is no...",None,"No, the Federal Work-Study (FWS) program is an...",1,1,0,1.680883,62793a53-f8d6-4a1b-a824-c5d115c6a2a1,20ecd716-a878-44c1-b6ab-0ac860f338af
9,What is Volume 8 about in relation to clinical...,Volume 8 discusses exceptions to the normal lo...,None,Inclusion of Clinical Work in a Standard Term ...,1,0,0,2.449734,ff7eabfd-ed7d-4d0d-bf5a-678bbe3f21e0,52960ada-1ad6-4d25-9743-32d0d0521bb3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [36]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [37]:
rag_documents = docs

In [38]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

---

#### ✅ Answer # 2:

Larger chunks can, in theory, provide more complete context by keeping full paragraphs/sections together. When related information is fragmented in smaller chunks, there is eroded coherence. However, small chunks could result in better precision because they can be more targeted containing more specific information and less noise. When chunks are too large, they burn a lot of tokens which can result in context erosion.

In [40]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

---

#### ✅ Answer # 3:

The embedding model is the foundation of the RAG system's retrieval capability. Using a larger embedding model (e.g. text-embedding-3-large vs text-embedding-3-small) allows for larger dimensionality, enabling more nuanced semantic relationships through better synonym recongition and more context awareness. However, larger embedding models are slower and more expensive, so choosing an embedding model is highly domain specific.

In [41]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [42]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [43]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [44]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided, there are several types of loans available to help with a student's cost of attendance:\n\n1. **Direct Subsidized Loans** – These loans are based on financial need, and the government pays the interest while the student is in school. Students may be eligible up to certain maximum amounts depending on their year and dependency status.\n\n2. **Direct Unsubsidized Loans** – These loans are available regardless of financial need, and interest accrues while the student is in school. Students can qualify for these loans in addition to subsidized loans.\n\n3. **Direct PLUS Loans** – These are loans that parents of dependent students can take out to help pay for their child's education costs if they meet eligibility requirements. There is no fixed loan limit for PLUS Loans, but they cannot exceed the student's cost of attendance minus other financial aid.\n\nI understand that navigating loans can feel overwhelming, and I hope thi

Finally, we can evaluate the new chain on the same test set!

In [45]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'essential-porter-93' at:
https://smith.langchain.com/o/4e8d6936-58a0-4ff9-bfae-6484988e5784/datasets/e39f7268-9973-4abb-a78e-c519c003076e/compare?selectedSessions=30c28ec9-a3c4-4b9f-83d4-75823a8a2a79




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,disbursement appendix A or B how does that wor...,Thank you for your thoughtful question about d...,None,The disbursement timing rules are explained in...,1,0,1,5.770279,23b41cf0-8073-411f-bc04-9737009339dc,029d6606-7450-4b30-9ac0-66c59e4aaf8f
1,How does the inclusion of clinical work in sta...,Thank you for your thoughtful question. Based ...,None,The inclusion of clinical work in standard ter...,1,1,1,7.902133,e28bd36e-054b-4441-b723-d3b9d4637bc8,0d4335a5-e619-4c96-8131-74f4ba0d70f0
2,Wha volume 2 and volume 8 discribe about the d...,Thank you for your thoughtful question. I unde...,None,Volume 2 explains the academic year requiremen...,0,0,1,6.939027,40bbabbe-19d2-4a0a-bb13-ed17bc811df1,48668de7-a429-45e4-ba52-9c5cf83b4c8e
3,Considering the detailed requirements outlined...,Thank you for your thoughtful question. Based ...,None,The regulations in Volume 2 specify that for c...,0,0,1,5.043196,866c5c78-eb89-4a97-97be-987d6cbaf8e0,c7711d4c-0fb5-47a9-8ebc-2bc09d27ac39
4,How credit hour allocation for clinical experi...,Thank you for your thoughtful question. Based ...,None,The context explains that clinical work includ...,1,1,1,8.045602,adf76b99-3f65-4bd8-9aca-d361f618b6cf,372a9b01-3eb3-4967-8b13-7e0f3fb8eccc
5,How do nonstandard terms and sequential course...,Thank you for your thoughtful question. It sou...,None,Nonstandard terms are defined as terms that do...,1,1,1,7.480447,8b260132-4e1f-4572-b001-c3aafcde7c98,38d532cc-93de-4dc4-9885-4111ede30101
6,"How do the regulatory citations, such as 34 CF...",Thank you for your thoughtful question. Based ...,None,The regulatory citations 34 CFR 668.3(a) and 3...,1,0,1,6.258179,4f048b64-5a62-4fea-80bd-b02ea951bb3c,5e21b094-c879-40c1-b536-6181ece0d7c9
7,How do the requirements for disbursement timin...,Thank you for your thoughtful question—navigat...,None,In clock-hour or non-term credit-hour programs...,1,0,1,6.241778,13443210-8a92-4356-9514-111fe686f5bc,46587f59-a398-4a85-8a1c-9b832c971951
8,Is the Federal Work-Study (FWS) program subjec...,Thank you for your thoughtful question. Based ...,None,"No, the Federal Work-Study (FWS) program is an...",1,1,1,2.768857,62793a53-f8d6-4a1b-a824-c5d115c6a2a1,3464e815-a9d6-43f8-8420-9a24c0ea2f1a
9,What is Volume 8 about in relation to clinical...,Thank you for your thoughtful question. Based ...,None,Inclusion of Clinical Work in a Standard Term ...,1,1,1,3.588462,ff7eabfd-ed7d-4d0d-bf5a-678bbe3f21e0,bd2c57b5-2ad2-4d68-9369-713d0abfd72d


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.


---

#### ✅ Response to Activity # 3:

The larger embedding model modestly improves both the correctness and helpfulness (by more directly answering the question), and our enhanced empathy rag prompt embeds empathetic langugage in now ALL of our query responses! The original query literally had 0% empathy while the updated query has 100% empathy.


![Comparison of RAG Chains](eval_comp.png)

